# Day 8: Conditional Routing — If This, Do That

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> A real agent doesn't do the same thing every time. It **routes** based on the input.

Conditional routing is the first pattern that makes an agent feel intelligent. Instead
of one agent handling everything, you have multiple specialists and a router that
decides which one to call.

## What You'll Build
- A router agent that classifies questions and delegates to specialists
- Math questions → calculator agent
- Geography questions → geography agent
- General questions → general agent

In [2]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 8 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (8 model(s) available)

DAY 8 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Today's Questions

1. **"What is 15% of 2340?"** → should route to math
2. **"What is the capital of Brazil?"** → should route to geography
3. **"What is the meaning of life?"** → should route to general
4. **"Calculate 20% of 5000 and tell me the capital of Japan."** → needs BOTH

---
## Framework 1: OpenAI Agents SDK — Handoffs

OpenAI SDK uses **handoffs** for routing. The router agent has `handoffs=[math_agent, geo_agent, general_agent]`
and the LLM decides which agent to transfer to. The `result.last_agent` attribute tells you
which specialist ended up handling the request.

In [ ]:
from agents import Agent, Runner, handoff, function_tool

model = get_openai_agents_model()

# ── Specialist Agents ──────────────────────────────────
@function_tool
def calculate(expression: str) -> str:
    """Evaluate a math expression. Handles percentages."""
    import re
    pct = re.match(r"(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)", expression)
    if pct:
        return str(float(pct.group(2)) * float(pct.group(1)) / 100)
    try:
        return str(eval(expression))
    except Exception:
        return f"Could not evaluate: {expression}"

math_agent = Agent(
    name="Math_Specialist",
    instructions="You solve math problems. Always use the calculate tool.",
    tools=[calculate],
    model=model,
)

geo_agent = Agent(
    name="Geography_Specialist",
    instructions="You answer geography questions. Give concise answers.",
    model=model,
)

general_agent = Agent(
    name="General_Assistant",
    instructions="You answer general knowledge questions thoughtfully.",
    model=model,
)

# ── Router Agent with handoffs ───────────────────────────
router = Agent(
    name="Router",
    instructions=(
        "You are a router. Classify the user's question:\n"
        "- Math questions (calculations, percentages, numbers) → hand off to Math_Specialist\n"
        "- Geography questions (capitals, countries, cities) → hand off to Geography_Specialist\n"
        "- Everything else → hand off to General_Assistant\n"
        "Hand off immediately. Do not answer the question yourself."
    ),
    handoffs=[math_agent, geo_agent, general_agent],
    model=model,
)

# ── Test routing ──────────────────────────────────────
questions = [
    "What is 15% of 2340?",
    "What is the capital of Brazil?",
    "What is the meaning of life?",
]

print("=" * 60)
print("OPENAI AGENTS SDK — CONDITIONAL ROUTING")
print("=" * 60)

for q in questions:
    print(f"\nQ: {q}")
    result = await Runner.run(router, q)
    print(f"Routed to: {result.last_agent.name}")
    print(f"A: {result.final_output[:200]}")

---
## Framework 2: LangGraph — Conditional Edges

LangGraph uses `add_conditional_edges()` for routing. A Python function inspects
the state and returns the name of the next node. This is the most explicit routing
of any framework.

In [4]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from typing_extensions import TypedDict, Annotated
import operator

model = get_langgraph_model()

# ── State ──────────────────────────────────────────────
class RouterState(TypedDict):
    messages: Annotated[list, operator.add]
    category: str

# ── Router node (LLM-based classification) ──────────────
def classify(state: RouterState) -> dict:
    messages = state["messages"]
    user_msg = messages[-1].content if messages else ""
    
    # Use LLM to classify
    response = model.invoke([
        SystemMessage(content="Classify this question as 'math', 'geography', or 'general'. Return ONLY the category word."),
        HumanMessage(content=user_msg),
    ])
    category = response.content.strip().lower()
    
    # Normalize
    for c in ["math", "geography", "general"]:
        if c in category:
            category = c
            break
    else:
        category = "general"
    
    return {"category": category}

# ── Specialist nodes ────────────────────────────────────
def handle_math(state: RouterState) -> dict:
    user_msg = state["messages"][-1].content
    response = model.invoke([
        SystemMessage(content="You are a math expert. Solve the problem step by step."),
        HumanMessage(content=user_msg),
    ])
    return {"messages": [response]}

def handle_geo(state: RouterState) -> dict:
    user_msg = state["messages"][-1].content
    response = model.invoke([
        SystemMessage(content="You are a geography expert. Be concise."),
        HumanMessage(content=user_msg),
    ])
    return {"messages": [response]}

def handle_general(state: RouterState) -> dict:
    user_msg = state["messages"][-1].content
    response = model.invoke([
        SystemMessage(content="You are a helpful assistant."),
        HumanMessage(content=user_msg),
    ])
    return {"messages": [response]}

# ── Routing function ────────────────────────────────────
def route(state: RouterState) -> str:
    return state.get("category", "general")

# ── Build the graph ────────────────────────────────────
builder = StateGraph(RouterState)
builder.add_node("classifier", classify)
builder.add_node("math_handler", handle_math)
builder.add_node("geo_handler", handle_geo)
builder.add_node("general_handler", handle_general)

builder.add_edge(START, "classifier")
builder.add_conditional_edges(
    "classifier",
    route,
    {"math": "math_handler", "geography": "geo_handler", "general": "general_handler"},
)
builder.add_edge("math_handler", END)
builder.add_edge("geo_handler", END)
builder.add_edge("general_handler", END)

graph = builder.compile()

# ── Test routing ──────────────────────────────────────
questions = [
    "What is 15% of 2340?",
    "What is the capital of Brazil?",
    "What is the meaning of life?",
]

print("=" * 60)
print("LANGGRAPH — CONDITIONAL ROUTING")
print("=" * 60)

for q in questions:
    result = graph.invoke({"messages": [HumanMessage(content=q)], "category": ""})
    print(f"\nQ: {q}")
    print(f"Routed to: {result['category']}")
    print(f"A: {result['messages'][-1].content[:200]}")

LANGGRAPH — CONDITIONAL ROUTING

Q: What is 15% of 2340?
Routed to: math
A: To find 15% of 2340, we can follow these steps:

Step 1: Convert the percentage to a decimal.
We know that 15% means 15 per 100. To convert it into a decimal form, divide 15 by 100:
\[ 15\% = \frac{15

Q: What is the capital of Brazil?
Routed to: geography
A: The capital of Brazil is Brasília.

Q: What is the meaning of life?
Routed to: general
A: The question "What is the meaning of life?" has been asked by philosophers, scientists, religious leaders, and thinkers for centuries, but no definitive answer has been universally accepted. Different


---
## Framework 3: CrewAI — Task Assignment

CrewAI handles routing through **developer-written rules**. A Python `if/elif` block
classifies the question by keywords and assigns the matching specialist agent to the
Crew's sequential process. The classification rules act as the router — unlike OpenAI
SDK and LangGraph, the LLM doesn't make the routing decision here.

In [5]:
from crewai import Agent, Task, Crew, Process

llm = get_crewai_llm()

# ── Router agent classifies then delegates ──────────────
router = Agent(
    role="Question Classifier",
    goal="Classify questions and pass to the right specialist",
    backstory="You are a smart router who categorizes questions into math, geography, or general.",
    llm=llm, verbose=True,
)

math_expert = Agent(
    role="Math Specialist",
    goal="Solve math problems accurately",
    backstory="You are a mathematician who loves solving calculations.",
    llm=llm, verbose=True,
)

geo_expert = Agent(
    role="Geography Specialist",
    goal="Answer geography questions precisely",
    backstory="You are a geographer with encyclopedic knowledge of countries and capitals.",
    llm=llm, verbose=True,
)

general_expert = Agent(
    role="General Knowledge Expert",
    goal="Answer general questions thoughtfully",
    backstory="You are a curious polymath who loves all types of questions.",
    llm=llm, verbose=True,
)

# ── Test routing ──────────────────────────────────────
questions = [
    "What is 15% of 2340?",
    "What is the capital of Brazil?",
    "What is the meaning of life?",
]

print("=" * 60)
print("CREWAI — CONDITIONAL ROUTING")
print("=" * 60)

for q in questions:
    classify_task = Task(
        description=f"Classify this question: '{q}'. Output: math, geography, or general.\nThen answer it.",
        expected_output=f"The category and the answer to: {q}",
        agent=router,
    )
    # Assign the specialist based on the question
    if "math" in q.lower() or any(c.isdigit() for c in q):
        specialist = math_expert
        category = "math"
    elif any(w in q.lower() for w in ["capital", "country", "city", "geography"]):
        specialist = geo_expert
        category = "geography"
    else:
        specialist = general_expert
        category = "general"
    
    answer_task = Task(
        description=f"Answer this {category} question: {q}",
        expected_output="The answer.",
        agent=specialist,
        context=[classify_task],
    )
    
    crew = Crew(agents=[router, specialist], tasks=[classify_task, answer_task],
                process=Process.sequential, verbose=False)
    result = await crew.kickoff_async()
    print(f"\nQ: {q}")
    print(f"Routed to: {category}")
    print(f"A: {str(result)[:200]}")

CREWAI — CONDITIONAL ROUTING


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Question Classifier                                                                                     │
│                                                                                                                 │
│  Task: Classify this question: 'What is 15% of 2340?'. Output: math, geography, or general.                     │
│  Then answer it.                                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Question Classifier                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The category is math. The answer to what is 15% of 2340 is 351.                                                │
│                                                                                                                 │
│  The actual complete content as the final answer is: The category is math and the answer to what is 15% of      │
│  2340 is 351.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Specialist                                                                                         │
│                                                                                                                 │
│  Task: Answer this math question: What is 15% of 2340?                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Question Classifier                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The meaning of life is categorized under the general classification. The question "What is the meaning of      │
│  life?" is too philosophical and existential to be classified into math or geography, thus it falls under       │
│  general.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: General Knowledge Expert                                                                                │
│                                                                                                                 │
│  Task: Answer this general question: What is the meaning of life?                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: General Knowledge Expert                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The concept of finding a single definitive answer to the meaning of life remains one of humanity's most        │
│  enduring questions without a widely accepted consensus. Different cultures, religions, and philosophies offer  │
│  diverse perspectives on this inquiry. Some might find purpose in personal fulfillment, community engagement,   │
│  or spiritual connection, while others believe that the meaning stems from self-improvement, contributing       │
│  positively to the world, or transcending material existence. Ultimately, what gives life meaning is deeply     │
│  individual and can evolve over time for each person as they navigate their unique circumstances and            │
│  experiences. Finding a universal answer is complex and might require more dialogue and understanding across    │
│  different cultural and philosophical perspectives.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Q: What is the meaning of life?
Routed to: general
A: The concept of finding a single definitive answer to the meaning of life remains one of humanity's most enduring questions without a widely accepted consensus. Different cultures, religions, and philo


---
## Comparison: Routing Approaches

| Framework | Routing mechanism | Who decides the route? | Visibility |
|---|---|---|---|
| OpenAI SDK | `handoffs=[agents]` on router | LLM decides | In Runner trace |
| LangGraph | `add_conditional_edges()` | Python function decides | Full graph visualization |
| CrewAI | Task assignment logic | Developer decides | In verbose logging |

**Key insight:** OpenAI SDK's handoffs are the most elegant — the LLM does the
classification naturally. LangGraph gives you the most explicit control — you can
see the exact routing function. CrewAI requires you to write the routing logic yourself.

## Key Takeaway

Routing is the first pattern that makes agents feel intelligent. The three approaches:
- OpenAI SDK: LLM-based handoffs (elegant, automatic)
- LangGraph: Conditional edges (explicit, debuggable)
- CrewAI: Task assignment (manual, predictable)

---

## LinkedIn Post Draft

> **Day 8: I taught an AI to route questions to the right specialist.**
>
> "What is 15% of 2340?" → math agent
> "What's the capital of Brazil?" → geography agent  
> "What is the meaning of life?" → general agent
>
> Three frameworks, three routing approaches:
> - OpenAI SDK: `handoffs=[math, geo, general]` — the LLM decides. Elegant.
> - LangGraph: `add_conditional_edges()` — a Python function decides. Explicit.
> - CrewAI: Task assignment logic — you decide. Predictable.
>
> OpenAI SDK wins on elegance. LangGraph wins on debuggability.  
> CrewAI wins when you need deterministic routing.
>
> #AIAgents #LangGraph #CrewAI #OpenAI #30DayChallenge